In [1]:
%pip install -U pip
%pip install pandas numpy scikit-learn tqdm

Looking in indexes: http://mirrors.aliyun.com/pypi/simple/
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: http://mirrors.aliyun.com/pypi/simple/
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import random
import hashlib
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

import mindspore as ms
from mindspore import context, nn, ops, Tensor

from sklearn.metrics import roc_auc_score, log_loss

# ------------------------------
# Runtime configuration
# ------------------------------
DEVICE_TARGET = os.getenv("MS_DEVICE_TARGET", "CPU")  # set to GPU explicitly if memory allows
context.set_context(mode=context.GRAPH_MODE, device_target=DEVICE_TARGET)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
ms.set_seed(SEED)
print("Using device target:", DEVICE_TARGET)

# ------------------------------
# Canonical paths (must be in SAME directory as this notebook runtime cwd).
# ------------------------------
# ------------------------------
# Canonical paths
# ------------------------------
NOTEBOOK_DIR = Path('/home/xceland/Desktop/Huawei/innovation/application/msmodels/workspace/dkt/update').resolve()

# Primary expected files (same directory)
TAXONOMY_PATH = NOTEBOOK_DIR / "cs_form3_form4_dkt_taxonomy.csv"
EVENTS_PATH = NOTEBOOK_DIR / "cs_dkt_events.csv"  # DB export in interaction_events-like shape

# Fallback lookup if files are not found in primary location
FALLBACK_DIRS = [
    Path.cwd().resolve(),
    Path('/home/xceland/Desktop/Huawei/innovation/application/zivAI_ASAG_ENGINE').resolve(),
]

if not TAXONOMY_PATH.exists():
    for d in FALLBACK_DIRS:
        candidate = d / "cs_form3_form4_dkt_taxonomy.csv"
        if candidate.exists():
            TAXONOMY_PATH = candidate
            break

if not EVENTS_PATH.exists():
    for d in FALLBACK_DIRS:
        candidate = d / "cs_dkt_events.csv"
        if candidate.exists():
            EVENTS_PATH = candidate
            break

ARTIFACT_DIR = NOTEBOOK_DIR
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

SKILL_MAP_VERSION = "v1"
SKILL_MAP_PATH = ARTIFACT_DIR / f"skill_map_{SKILL_MAP_VERSION}.json"
MODEL_META_PATH = ARTIFACT_DIR / "model_meta.json"

assert TAXONOMY_PATH.exists(), (
    f"Missing taxonomy file: {TAXONOMY_PATH}\n"
    f"Checked NOTEBOOK_DIR={NOTEBOOK_DIR} and fallbacks={FALLBACK_DIRS}"
)
assert EVENTS_PATH.exists(), (
    f"Missing training events file: {EVENTS_PATH}\n"
    f"Checked NOTEBOOK_DIR={NOTEBOOK_DIR} and fallbacks={FALLBACK_DIRS}"
)

print("MindSpore:", ms.__version__)
print("Device:", ms.get_context("device_target"))
print("Taxonomy:", TAXONOMY_PATH)
print("Events:", EVENTS_PATH)
print("Output dir:", ARTIFACT_DIR)


Using device target: CPU
MindSpore: 1.8.0
Device: CPU
Taxonomy: /workspace/workspace/dkt/update/cs_form3_form4_dkt_taxonomy.csv
Events: /workspace/workspace/dkt/update/cs_dkt_events.csv
Output dir: /home/xceland/Desktop/Huawei/innovation/application/msmodels/workspace/dkt/update


Load DB-exported event data (interaction_events-style) and validate contract.

Required columns:
- student_id
- skill_code
- is_correct (0/1)
- event_time

Optional but useful:
- subject_code / subject_id
- score / max_score
- attempt_id / question_id



In [3]:
events = pd.read_csv(EVENTS_PATH)

required_cols = ["student_id", "skill_code", "is_correct", "event_time"]
missing = [c for c in required_cols if c not in events.columns]
assert not missing, f"Missing required columns in events CSV: {missing}"

# Normalize data types
for c in ["student_id", "skill_code", "event_time"]:
    events[c] = events[c].astype(str).str.strip()

events = events[(events["student_id"] != "") & (events["skill_code"] != "")]

# Correctness must be binary
if events["is_correct"].dtype == object:
    events["is_correct"] = events["is_correct"].astype(str).str.strip().str.lower().map(
        {"1": 1, "0": 0, "true": 1, "false": 0, "yes": 1, "no": 0}
    )

events["is_correct"] = pd.to_numeric(events["is_correct"], errors="coerce")
events = events.dropna(subset=["is_correct"])
events["is_correct"] = events["is_correct"].astype(int).clip(0, 1)

# Build richer outcome state for input tokens: 0=wrong, 1=partial, 2=full/correct
if {"score", "max_score"}.issubset(events.columns):
    events["score"] = pd.to_numeric(events["score"], errors="coerce").fillna(0.0)
    events["max_score"] = pd.to_numeric(events["max_score"], errors="coerce").replace(0, np.nan)
    ratio = (events["score"] / events["max_score"]).fillna(0.0).clip(0.0, 1.0)
    events["outcome_state"] = np.where(ratio >= 0.85, 2, np.where(ratio >= 0.35, 1, 0)).astype(int)
    # If marked fully correct, force full outcome state.
    events.loc[events["is_correct"] == 1, "outcome_state"] = 2
else:
    # Fallback when score columns are absent
    events["outcome_state"] = (events["is_correct"] * 2).astype(int)

# Parse event time
events["event_time"] = pd.to_datetime(events["event_time"], errors="coerce", utc=True)
events = events.dropna(subset=["event_time"]).copy()

# Optional: keep only Computer Science if subject column exists
if "subject_code" in events.columns:
    events["subject_code"] = events["subject_code"].astype(str).str.strip().str.lower()
    events = events[events["subject_code"].isin(["computer_science", "cs", "comp_sci"]) | (events["subject_code"] == "")]

# Stable ordering keys for sequential modeling
sort_cols = ["student_id", "event_time"]
for extra in ["attempt_id", "question_id", "id"]:
    if extra in events.columns:
        sort_cols.append(extra)

events = events.sort_values(sort_cols).reset_index(drop=True)

print("Events rows after cleaning:", len(events))
print("Unique students:", events["student_id"].nunique())
print(events[["student_id", "skill_code", "is_correct", "event_time"]].head())


Events rows after cleaning: 100000
Unique students: 578
  student_id                                        skill_code  is_correct  \
0   stu_0001  CS.F4.TECH.IDENTIFY_BUSINESS_LOCATION_CONDITIONS           0   
1   stu_0001  CS.F4.TECH.IDENTIFY_BUSINESS_LOCATION_CONDITIONS           1   
2   stu_0001                CS.F4.TECH.CONDUCT_MARKET_RESEARCH           1   
3   stu_0001  CS.F4.TECH.IDENTIFY_BUSINESS_LOCATION_CONDITIONS           1   
4   stu_0001  CS.F4.TECH.IDENTIFY_BUSINESS_LOCATION_CONDITIONS           0   

                 event_time  
0 2024-03-23 11:52:00+00:00  
1 2024-03-23 11:58:00+00:00  
2 2024-03-23 12:11:00+00:00  
3 2024-03-23 12:20:00+00:00  
4 2024-03-23 12:32:00+00:00  


Freeze skill indices from syllabus taxonomy (do not rebuild from raw events).

- Build deterministic mapping once from taxonomy.
- Save mapping artifact per model version.
- Reuse same mapping for all future training/inference of that model version.



In [4]:
def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def save_json(path: Path, obj: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=True), encoding="utf-8")

# Load curriculum taxonomy
taxonomy = pd.read_csv(TAXONOMY_PATH)
required_tax_cols = ["skill_code", "dkt_trackable", "active"]
missing_tax = [c for c in required_tax_cols if c not in taxonomy.columns]
assert not missing_tax, f"Missing required taxonomy columns: {missing_tax}"

for c in ["skill_code"]:
    taxonomy[c] = taxonomy[c].astype(str).str.strip()

taxonomy = taxonomy[(taxonomy["active"] == 1) & (taxonomy["dkt_trackable"] == 1)]
taxonomy = taxonomy[taxonomy["skill_code"] != ""].copy()

canonical_skill_codes = sorted(taxonomy["skill_code"].unique().tolist())
assert canonical_skill_codes, "No active trackable skills found in taxonomy."

taxonomy_fingerprint = sha256_text("\n".join(canonical_skill_codes))

if SKILL_MAP_PATH.exists():
    saved_map = json.loads(SKILL_MAP_PATH.read_text(encoding="utf-8"))
    saved_codes = sorted(saved_map["skill_code_to_idx"].keys())

    if saved_codes != canonical_skill_codes:
        raise ValueError(
            "Frozen skill map does not match current taxonomy. "
            "Create a NEW mapping/model version instead of mutating indices."
        )

    skill_to_idx = {k: int(v) for k, v in saved_map["skill_code_to_idx"].items()}
    print(f"Loaded frozen mapping: {SKILL_MAP_PATH}")
else:
    skill_to_idx = {code: idx for idx, code in enumerate(canonical_skill_codes)}

    save_json(SKILL_MAP_PATH, {
        "mapping_version": SKILL_MAP_VERSION,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "taxonomy_path": str(TAXONOMY_PATH),
        "taxonomy_fingerprint": taxonomy_fingerprint,
        "num_skills": len(skill_to_idx),
        "skill_code_to_idx": skill_to_idx,
    })
    print(f"Created new frozen mapping: {SKILL_MAP_PATH}")

# Reverse mapping by index
idx_to_skill = [None] * len(skill_to_idx)
for code, idx in skill_to_idx.items():
    idx_to_skill[idx] = code

K = len(skill_to_idx)

# Validate events against frozen mapping
unknown_mask = ~events["skill_code"].isin(skill_to_idx)
unknown_n = int(unknown_mask.sum())
if unknown_n > 0:
    samples = events.loc[unknown_mask, "skill_code"].drop_duplicates().head(20).tolist()
    raise ValueError(
        f"Found {unknown_n} rows with unknown skill_code not in frozen taxonomy map. "
        f"Examples: {samples}"
    )

events["skill_idx"] = events["skill_code"].map(skill_to_idx).astype(int)

print("Num skills (K):", K)
print("First 10 skills:", idx_to_skill[:10])


Loaded frozen mapping: /home/xceland/Desktop/Huawei/innovation/application/msmodels/workspace/dkt/update/skill_map_v1.json
Num skills (K): 77
First 10 skills: ['CS.F3.ALG.APPLY_TOP_DOWN', 'CS.F3.ALG.CONSTRUCT_PSEUDOCODE', 'CS.F3.ALG.DEBUG_ALGORITHMS', 'CS.F3.ALG.DESIGN_FLOWCHARTS', 'CS.F3.ALG.USE_TRACE_TABLES', 'CS.F3.APP.DESCRIBE_APPLICATION_AREAS', 'CS.F3.DATA.CONVERT_DENARY_TO_HEX', 'CS.F3.DATA.CONVERT_DENARY_TO_OCTAL', 'CS.F3.DATA.CONVERT_HEX_TO_DENARY', 'CS.F3.DATA.CONVERT_OCTAL_TO_DENARY']


Build per-student ordered interaction sequences.
Each event is one question-level observation.



In [5]:
user_seqs = []
user_ids = []

for uid, g in events.groupby("student_id", sort=False):
    skills = g["skill_idx"].to_numpy(np.int32)
    outcomes = g["outcome_state"].to_numpy(np.int32)
    corr = g["is_correct"].to_numpy(np.int32)

    # Need at least 3 events for next-step prediction signal
    if len(skills) < 3:
        continue

    user_ids.append(uid)
    user_seqs.append((skills, outcomes, corr))

assert len(user_seqs) > 0, "No usable student sequences found."

lengths = np.array([len(s[0]) for s in user_seqs], dtype=np.int32)
print("Sequences kept:", len(user_seqs))
print("Min/Median/Max length:", int(lengths.min()), float(np.median(lengths)), int(lengths.max()))

# Train/Val/Test split by student (no leakage)
uids = np.array(user_ids)
rng = np.random.default_rng(SEED)
perm = rng.permutation(len(uids))

n = len(uids)
n_train = int(0.8 * n)
n_val = int(0.1 * n)

train_idx = perm[:n_train]
val_idx = perm[n_train:n_train + n_val]
test_idx = perm[n_train + n_val:]

train_seqs = [user_seqs[i] for i in train_idx]
val_seqs = [user_seqs[i] for i in val_idx]
test_seqs = [user_seqs[i] for i in test_idx]

print("Train/Val/Test users:", len(train_seqs), len(val_seqs), len(test_seqs))


Sequences kept: 578
Min/Median/Max length: 58 172.0 230
Train/Val/Test users: 462 57 59


Windowing + dataset generator (DKT standard).

Tokenization convention (must stay identical for train + inference):
- PAD token = 0
- interaction token = skill_idx + is_correct * K + 1
  (+1 offset reserves 0 strictly for PAD)



Targets per timestep:
- next_skill (shifted skills)
- next_correct (shifted correctness)
- mask (ignore PAD targets)



In [6]:
MAX_LEN = 64
STRIDE = 20
BATCH_SIZE = 8

# Moderate model upgrade: stacked LSTM + dropout
EMB_DIM = 96
HIDDEN_DIM = 128
LSTM_LAYERS = 1
LSTM_DROPOUT = 0.00
HEAD_DROPOUT = 0.10

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-5
MIN_LR_FACTOR = 0.35
EPOCHS = 35
PATIENCE = 6

print("Effective LSTM dropout:", LSTM_DROPOUT)

PAD_X = 0
PAD_SKILL = -1

def make_windows(skills, corr, max_len, stride):
    L = len(skills)
    out = []
    start = 0
    while start < L:
        end = min(start + max_len, L)
        out.append((skills[start:end], corr[start:end]))
        if end == L:
            break
        start += stride
    return out

def pad_seq(arr, max_len, pad_val):
    out = np.full((max_len,), pad_val, dtype=np.int32)
    out[:len(arr)] = arr
    return out

def gen_samples(seqs, max_len):
    for skills, outcomes, corr in seqs:
        sw = make_windows(skills, corr, max_len, STRIDE)
        ow = make_windows(outcomes, corr, max_len, STRIDE)
        cw = make_windows(corr, corr, max_len, STRIDE)

        for i in range(min(len(sw), len(ow), len(cw))):
            ws = sw[i][0]
            wo = ow[i][0]
            wc = cw[i][0]

            if len(ws) < 2:
                continue

            # IMPORTANT: +1 offset to keep PAD=0 unique
            # token = skill + (outcome_state * K) + 1, where outcome_state in {0,1,2}
            x_tok = (ws + wo * K + 1).astype(np.int32)
            x_tok = pad_seq(x_tok, max_len, pad_val=PAD_X)

            next_skill = pad_seq(ws[1:].astype(np.int32), max_len, pad_val=PAD_SKILL)
            next_corr = pad_seq(wc[1:].astype(np.int32), max_len, pad_val=0)
            mask = (next_skill != PAD_SKILL).astype(np.float32)

            yield x_tok, next_skill, next_corr, mask

tmp = next(gen_samples(train_seqs[:1], MAX_LEN))
print([a.shape for a in tmp], "mask sum:", float(tmp[-1].sum()))


Effective LSTM dropout: 0.0
[(64,), (64,), (64,), (64,)] mask sum: 63.0


In [7]:
import mindspore.dataset as ds

ds.config.set_num_parallel_workers(1)
ds.config.set_prefetch_size(2)
ds.config.set_enable_shared_mem(False)

def build_dataset(seqs, batch_size, max_len, shuffle=True):
    columns = ["x_tok", "next_skill", "next_corr", "mask"]
    dataset = ds.GeneratorDataset(
        source=lambda: gen_samples(seqs, max_len),
        column_names=columns,
        shuffle=shuffle,
        num_parallel_workers=1,
        python_multiprocessing=False,
    )
    # Keep remainder to avoid silent data loss
    dataset = dataset.batch(batch_size, drop_remainder=False)
    return dataset

train_ds = build_dataset(train_seqs, BATCH_SIZE, MAX_LEN, shuffle=True)
val_ds = build_dataset(val_seqs, BATCH_SIZE, MAX_LEN, shuffle=False)
test_ds = build_dataset(test_seqs, BATCH_SIZE, MAX_LEN, shuffle=False)

print("Train batches:", train_ds.get_dataset_size())


Train batches: 400


Standard DKT-LSTM model producing per-timestep per-skill logits: (B, T, K).



In [8]:
class DKTNetLSTM(nn.Cell):
    def __init__(self, num_skills, emb_dim=96, hidden_dim=128, dropout=0.0, num_layers=1, head_dropout=0.10):
        super().__init__()
        self.K = num_skills
        self.vocab = 3 * num_skills + 1  # +1 PAD token; 3 outcome states
        self.emb = nn.Embedding(self.vocab, emb_dim, padding_idx=0)

        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            has_bias=True,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.head = nn.SequentialCell([
            nn.Dense(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(keep_prob=1.0 - head_dropout),
            nn.Dense(hidden_dim, num_skills),
        ])

    def construct(self, x_tok):
        x = self.emb(x_tok)
        h, _ = self.lstm(x)
        logits = self.head(h)
        return logits

model = DKTNetLSTM(
    K,
    emb_dim=EMB_DIM,
    hidden_dim=HIDDEN_DIM,
    dropout=LSTM_DROPOUT,
    num_layers=LSTM_LAYERS,
    head_dropout=HEAD_DROPOUT,
)
print(model)


DKTNetLSTM<
  (emb): Embedding<vocab_size=232, embedding_size=96, use_one_hot=False, embedding_table=Parameter (name=emb.embedding_table, shape=(232, 96), dtype=Float32, requires_grad=True), dtype=Float32, padding_idx=0>
  (lstm): LSTM<
    (rnn): _DynamicLSTMCPUGPU<>
    (reverse): _Reverse<>
    (reverse_sequence): _ReverseSequence<>
    (dropout_op): Dropout<keep_prob=1.0>
    >
  (head): SequentialCell<
    (0): Dense<input_channels=128, output_channels=128, has_bias=True>
    (1): ReLU<>
    (2): Dropout<keep_prob=0.9>
    (3): Dense<input_channels=128, output_channels=77, has_bias=True>
    >
  >


Masked next-skill BCE loss:
- p = sigmoid(logits[b,t,next_skill[b,t]])
- BCE vs next_correct
- average over valid (non-pad) targets only



In [9]:
sigmoid = ops.Sigmoid()
gather = ops.GatherD()
eps = 1e-7

class DKTLossCell(nn.Cell):
    def __init__(self, net):
        super().__init__()
        self.net = net

    def construct(self, x_tok, next_skill, next_corr, mask):
        logits = self.net(x_tok)
        probs = sigmoid(logits)

        safe_skill = ops.maximum(next_skill, Tensor(0, ms.int32))
        idx = ops.ExpandDims()(safe_skill, -1)
        p = gather(probs, 2, idx).squeeze(-1)

        y = next_corr.astype(ms.float32)
        p = ops.clip_by_value(p, eps, 1 - eps)

        bce = -(y * ops.log(p) + (1 - y) * ops.log(1 - p))
        loss = (bce * mask).sum() / (mask.sum() + eps)
        return loss

loss_cell = DKTLossCell(model)
steps_per_epoch = train_ds.get_dataset_size()
total_steps = max(1, steps_per_epoch * EPOCHS)
lr_array = np.linspace(LEARNING_RATE, LEARNING_RATE * MIN_LR_FACTOR, total_steps, dtype=np.float32)
lr_schedule = Tensor(lr_array, ms.float32)

optimizer = nn.Adam(
    model.trainable_params(),
    learning_rate=lr_schedule,
    weight_decay=WEIGHT_DECAY,
)
train_step = nn.TrainOneStepCell(loss_cell, optimizer)
train_step.set_train()


TrainOneStepCell<
  (network): DKTLossCell<
    (net): DKTNetLSTM<
      (emb): Embedding<vocab_size=232, embedding_size=96, use_one_hot=False, embedding_table=Parameter (name=net.emb.embedding_table, shape=(232, 96), dtype=Float32, requires_grad=True), dtype=Float32, padding_idx=0>
      (lstm): LSTM<
        (rnn): _DynamicLSTMCPUGPU<>
        (reverse): _Reverse<>
        (reverse_sequence): _ReverseSequence<>
        (dropout_op): Dropout<keep_prob=1.0>
        >
      (head): SequentialCell<
        (0): Dense<input_channels=128, output_channels=128, has_bias=True>
        (1): ReLU<>
        (2): Dropout<keep_prob=0.9>
        (3): Dense<input_channels=128, output_channels=77, has_bias=True>
        >
      >
    >
  (optimizer): Adam<
    (learning_rate): _IteratorLearningRate<>
    >
  >

In [10]:
batch = next(train_ds.create_dict_iterator())
x_tok = batch["x_tok"]
print("x_tok shape:", x_tok.shape)

model.set_train(False)
_ = model(x_tok)
print("Forward pass OK")


x_tok shape: (8, 64)
Forward pass OK


Training + validation loop (AUC, accuracy, logloss).



In [11]:
def eval_epoch(net, dataset, max_batches=None):
    net.set_train(False)
    all_p, all_y = [], []

    for bi, batch in enumerate(dataset.create_dict_iterator()):
        if max_batches is not None and bi >= max_batches:
            break

        x_tok = batch["x_tok"]
        ns = batch["next_skill"].asnumpy().astype(np.int32)
        y = batch["next_corr"].asnumpy().astype(np.float32)
        m = batch["mask"].asnumpy().astype(np.float32)

        probs = sigmoid(net(x_tok)).asnumpy().astype(np.float32)
        ns_clip = np.clip(ns, 0, K - 1)
        p = np.take_along_axis(probs, ns_clip[..., None], axis=2).squeeze(2)

        valid = m > 0
        all_p.append(p[valid])
        all_y.append(y[valid])

    if not all_p:
        return {"auc": np.nan, "acc": np.nan, "logloss": np.nan}

    p = np.concatenate(all_p)
    y = np.concatenate(all_y)

    auc = roc_auc_score(y, p) if len(np.unique(y)) > 1 else np.nan
    acc = float(((p >= 0.5) == (y >= 0.5)).mean())
    ll = float(log_loss(y, np.clip(p, 1e-7, 1 - 1e-7)))

    return {"auc": float(auc), "acc": acc, "logloss": ll}


In [12]:
CKPT_DIR = ARTIFACT_DIR / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

BEST_CKPT = CKPT_DIR / "best_dkt_lstm_v3.ckpt"
print("Saving checkpoints to:", BEST_CKPT)

best_auc = -1.0
bad = 0

for epoch in range(1, EPOCHS + 1):
    model.set_train(True)
    losses = []

    for batch in tqdm(train_ds.create_dict_iterator(), total=train_ds.get_dataset_size(), desc=f"Epoch {epoch}"):
        loss = train_step(batch["x_tok"], batch["next_skill"], batch["next_corr"], batch["mask"])
        losses.append(float(loss.asnumpy()))

    val_metrics = eval_epoch(model, val_ds)
    print(
        f"\nEpoch {epoch} | train_loss={np.mean(losses):.4f} | "
        f"val_auc={val_metrics['auc']:.4f} | val_acc={val_metrics['acc']:.4f} | "
        f"val_logloss={val_metrics['logloss']:.4f}\n"
    )

    if val_metrics["auc"] > best_auc + 1e-4:
        best_auc = val_metrics["auc"]
        bad = 0
        ms.save_checkpoint(model, str(BEST_CKPT))
        print(f"Saved best checkpoint to {BEST_CKPT} (auc={best_auc:.4f})")
    else:
        bad += 1
        if bad >= PATIENCE:
            print(f"Early stopping (best auc={best_auc:.4f})")
            break

ms.load_checkpoint(str(BEST_CKPT), net=model)
test_metrics = eval_epoch(model, test_ds)
print("TEST(best):", test_metrics)

model_meta = {
    "saved_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_type": "dkt_lstm_v3",
    "skill_map_version": SKILL_MAP_VERSION,
    "skill_map_path": str(SKILL_MAP_PATH),
    "num_skills": K,
    "max_len": MAX_LEN,
    "stride": STRIDE,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "architecture": {
        "embedding_dim": EMB_DIM,
        "hidden_dim": HIDDEN_DIM,
        "lstm_layers": LSTM_LAYERS,
        "lstm_dropout": LSTM_DROPOUT,
        "head_dropout": HEAD_DROPOUT,
    },
    "checkpoint_path": str(BEST_CKPT),
    "metrics": test_metrics,
}
MODEL_META_PATH.write_text(json.dumps(model_meta, indent=2, ensure_ascii=True), encoding="utf-8")
print("Saved model metadata:", MODEL_META_PATH)


Saving checkpoints to: /home/xceland/Desktop/Huawei/innovation/application/msmodels/workspace/dkt/update/checkpoints/best_dkt_lstm_v3.ckpt


Epoch 1: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 400/400 [00:22<00:00, 17.40it/s]



Epoch 1 | train_loss=0.6928 | val_auc=0.5458 | val_acc=0.5509 | val_logloss=0.6873

Saved best checkpoint to /home/xceland/Desktop/Huawei/innovation/application/msmodels/workspace/dkt/update/checkpoints/best_dkt_lstm_v3.ckpt (auc=0.5458)


Epoch 2: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 400/400 [00:18<00:00, 21.91it/s]



Epoch 2 | train_loss=0.6671 | val_auc=0.6694 | val_acc=0.6372 | val_logloss=0.6370

Saved best checkpoint to /home/xceland/Desktop/Huawei/innovation/application/msmodels/workspace/dkt/update/checkpoints/best_dkt_lstm_v3.ckpt (auc=0.6694)


...


Epoch 33 | train_loss=0.6319 | val_auc=0.6786 | val_acc=0.6415 | val_logloss=0.6304



Epoch 34: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 400/400 [00:06<00:00, 57.72it/s]



Epoch 34 | train_loss=0.6316 | val_auc=0.6786 | val_acc=0.6418 | val_logloss=0.6304



Epoch 35: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 400/400 [00:06<00:00, 63.50it/s]



Epoch 35 | train_loss=0.6315 | val_auc=0.6786 | val_acc=0.6418 | val_logloss=0.6304

TEST(best): {'auc': 0.714077695350142, 'acc': 0.671622994109283, 'logloss': 0.6158544743013241}
Saved model metadata: /home/xceland/Desktop/Huawei/innovation/application/msmodels/workspace/dkt/update/model_meta.json


Cloud + Edge model packaging

- Cloud model: full checkpoint for server inference/training continuation.
- Edge model: exported MindIR for MindSpore Lite conversion/deployment.


In [13]:
# Confirm final test metrics
test_metrics = eval_epoch(model, test_ds)
print("TEST:", test_metrics)

# Cloud artifact (full model checkpoint)
CLOUD_CKPT_PATH = ARTIFACT_DIR / "dkt_lstm_v3_cloud.ckpt"
ms.save_checkpoint(model, str(CLOUD_CKPT_PATH))
print("Saved cloud checkpoint:", CLOUD_CKPT_PATH)

# Edge artifact (MindIR for MindSpore Lite conversion)
EDGE_MINDIR_BASENAME = ARTIFACT_DIR / "dkt_lstm_v3_edge"
DUMMY_INPUT = Tensor(np.zeros((1, MAX_LEN), dtype=np.int32), ms.int32)
ms.export(model, DUMMY_INPUT, file_name=str(EDGE_MINDIR_BASENAME), file_format="MINDIR")
EDGE_MINDIR_PATH = EDGE_MINDIR_BASENAME.with_suffix(".mindir")
print("Saved edge MindIR:", EDGE_MINDIR_PATH)

# Optional: smoke test with mindspore_lite if available
try:
    import mindspore_lite as mslite  # type: ignore
    print("mindspore_lite detected:", mslite.__version__ if hasattr(mslite, "__version__") else "ok")
    print("You can convert/run on edge using this MindIR.")
except Exception:
    print("mindspore_lite not installed in this environment; export completed for later conversion.")

print("Suggested converter command (run where converter_lite is available):")
print(
    f"converter_lite --fmk=MINDIR --modelFile={EDGE_MINDIR_PATH} "
    f"--outputFile={ARTIFACT_DIR / 'dkt_lstm_v3_edge_lite'} --optimize=ascend_oriented"
)

# update metadata with packaging outputs
meta = json.loads(MODEL_META_PATH.read_text(encoding='utf-8'))
meta["cloud_checkpoint_path"] = str(CLOUD_CKPT_PATH)
meta["edge_mindir_path"] = str(EDGE_MINDIR_PATH)
MODEL_META_PATH.write_text(json.dumps(meta, indent=2, ensure_ascii=True), encoding='utf-8')
print("Updated model metadata with cloud/edge artifacts:", MODEL_META_PATH)


TEST: {'auc': 0.714077695350142, 'acc': 0.671622994109283, 'logloss': 0.6158544743013241}
Saved cloud checkpoint: /home/xceland/Desktop/Huawei/innovation/application/msmodels/workspace/dkt/update/dkt_lstm_v3_cloud.ckpt
Saved edge MindIR: /home/xceland/Desktop/Huawei/innovation/application/msmodels/workspace/dkt/update/dkt_lstm_v3_edge.mindir
mindspore_lite not installed in this environment; export completed for later conversion.
Suggested converter command (run where converter_lite is available):
converter_lite --fmk=MINDIR --modelFile=/home/xceland/Desktop/Huawei/innovation/application/msmodels/workspace/dkt/update/dkt_lstm_v3_edge.mindir --outputFile=/home/xceland/Desktop/Huawei/innovation/application/msmodels/workspace/dkt/update/dkt_lstm_v3_edge_lite --optimize=ascend_oriented
Updated model metadata with cloud/edge artifacts: /home/xceland/Desktop/Huawei/innovation/application/msmodels/workspace/dkt/update/model_meta.json


Inference helpers:
- predict_next_correct: P(correct on target skill)
- mastery_vector_from_history: per-skill mastery at current state

Both use the SAME frozen tokenization as training (+1 offset, PAD=0).



In [14]:
def _encode_history_tokens(history_skills, history_outcomes, max_len=MAX_LEN):
    hs = np.asarray(history_skills, dtype=np.int32)
    ho = np.asarray(history_outcomes, dtype=np.int32)

    assert hs.shape == ho.shape, "history_skills and history_outcomes must have same length"

    if len(hs) == 0:
        return np.zeros((1, max_len), dtype=np.int32), 0

    hs = hs[-max_len:]
    ho = ho[-max_len:]
    valid_len = len(hs)

    # IMPORTANT: same convention used during training
    x_tok = hs + ho * K + 1
    x_tok = np.pad(x_tok, (0, max_len - valid_len), constant_values=0)

    return x_tok.reshape(1, -1).astype(np.int32), valid_len


def predict_next_correct(net, history_skills, history_outcomes, next_skill_idx, max_len=MAX_LEN):
    net.set_train(False)
    x_tok_np, valid_len = _encode_history_tokens(history_skills, history_outcomes, max_len=max_len)

    if valid_len == 0:
        return 0.5

    x_tok = Tensor(x_tok_np, ms.int32)
    probs = sigmoid(net(x_tok)).asnumpy()

    t = valid_len - 1  # last non-pad timestep
    return float(probs[0, t, int(next_skill_idx)])


def mastery_vector_from_history(net, history_skills, history_outcomes, max_len=MAX_LEN):
    net.set_train(False)
    x_tok_np, valid_len = _encode_history_tokens(history_skills, history_outcomes, max_len=max_len)

    if valid_len == 0:
        return np.full((K,), 0.5, dtype=np.float32)

    x_tok = Tensor(x_tok_np, ms.int32)
    probs = sigmoid(net(x_tok)).asnumpy()

    t = valid_len - 1
    return probs[0, t, :].astype(np.float32)


# Example quick check with first test sequence
if len(test_seqs) > 0:
    skills, outcomes, corr = test_seqs[0]
    p = predict_next_correct(model, skills[:-1], outcomes[:-1], int(skills[-1]))
    print("Predicted P(correct next):", p, "| true:", int(corr[-1]))


Predicted P(correct next): 0.1187330111861229 | true: 0


In [15]:
# Service-style response example (top weak skills)
if len(test_seqs) > 0:
    skills, outcomes, corr = test_seqs[0]
    mastery = mastery_vector_from_history(model, skills, outcomes)

    weak_idx = np.argsort(mastery)[:5]
    weak = [
        {
            "skill_code": idx_to_skill[int(i)],
            "mastery_prob": float(mastery[int(i)]),
        }
        for i in weak_idx
    ]

    avg_mastery = float(mastery.mean())
    risk_level = "high" if avg_mastery < 0.45 else ("medium" if avg_mastery < 0.70 else "low")

    dkt_response = {
        "student_id": "demo_student",
        "snapshot_time_utc": datetime.now(timezone.utc).isoformat(),
        "average_mastery": avg_mastery,
        "risk_level": risk_level,
        "weak_skills": weak,
        "model_meta_path": str(MODEL_META_PATH),
    }

    print(json.dumps(dkt_response, indent=2))


{
  "student_id": "demo_student",
  "snapshot_time_utc": "2026-03-06T01:07:35.988934+00:00",
  "average_mastery": 0.16463503241539001,
  "risk_level": "high",
  "weak_skills": [
    {
      "skill_code": "CS.F3.SEC.SETUP_FIREWALL",
      "mastery_prob": 0.08620008826255798
    },
    {
      "skill_code": "CS.F3.DATA.STORAGE_UNITS_EXPLAIN",
      "mastery_prob": 0.08974260091781616
    },
    {
      "skill_code": "CS.F4.TECH.IDENTIFY_BUSINESS_LOCATION_CONDITIONS",
      "mastery_prob": 0.0903826579451561
    },
    {
      "skill_code": "CS.F3.SAD.DESIGN_INPUT_OUTPUT_UI",
      "mastery_prob": 0.09136335551738739
    },
    {
      "skill_code": "CS.F4.APP.DESIGN_DOMAIN_MODELS",
      "mastery_prob": 0.09701859951019287
    }
  ],
  "model_meta_path": "/home/xceland/Desktop/Huawei/innovation/application/msmodels/workspace/dkt/update/model_meta.json"
}
